# Семинар (80 минут): Fine-tuning BERT для Named Entity Recognition (NER)

## Формат семинара
Структура повторяется по всем разделам:

1. Теория (кратко, но формально)
2. Демонстрация (пример с кодом)
3. Самостоятельная работа (задача)
4. Эталонное решение (сразу после задачи)

## Цели
К концу семинара вы должны уметь:
- объяснить постановку NER как задачи sequence labeling;
- использовать BIO-разметку и корректно выравнивать метки после токенизации;
- дообучить BERT-подобную модель для NER с помощью HuggingFace Transformers;
- оценить качество с помощью метрики F1 (seqeval) и выполнить инференс на своих примерах.

## План по времени (ориентир)
- 0–10: Постановка NER и BIO, что такое fine-tuning для token classification
- 10–25: Данные и формат разметки; выравнивание BIO-меток после subword-токенизации
- 25–55: Fine-tuning BERT (Trainer), логирование, валидация
- 55–70: Оценка качества (Precision/Recall/F1), анализ ошибок
- 70–80: Инференс на своих примерах, итоговые вопросы

## Важно про окружение

Этот блокнот рассчитан на стандартную установку Python с библиотеками:
- `transformers`
- `datasets`
- `evaluate`
- `seqeval`
- `torch`

В некоторых средах (например, без доступа к интернету) скачивание датасета и предобученной модели может быть недоступно.
В блокноте предусмотрены fallback-варианты:
- если датасет `conll2003` недоступен, используется маленький встроенный toy-датасет;
- если модель/токенизатор не удаётся скачать, будет предложено указать локальный путь к ранее скачанным файлам.

In [ ]:

# Установка зависимостей (при необходимости)
# Если у вас уже всё установлено, эту ячейку можно пропустить.

# !pip -q install transformers datasets evaluate seqeval accelerate

import os
import random
import numpy as np

import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Torch:", torch.__version__)

# 1. Теория: NER как sequence labeling и BIO-разметка (0–10 минут)

## Постановка
В NER мы решаем две задачи одновременно:
1) найти границы сущностей (segmentation),
2) определить тип сущности (classification).

Чтобы свести задачу к "один тег на токен", применяют BIO-разметку:

- **B-TYPE**: начало сущности типа TYPE
- **I-TYPE**: продолжение сущности типа TYPE
- **O**: вне сущности

Пример:

Текст: `John lives in New York`
Теги: `B-PER O O B-LOC I-LOC`

## Почему NER сложнее POS
В POS каждый токен получает тег, а границы заранее фиксированы.
В NER границы сущностей нужно найти — отсюда дополнительная сложность.

## Метрика
Обычно оценивают **Precision/Recall/F1** по целым сущностям, а не по отдельным токенам (используют `seqeval`).

# 2. Данные: формат разметки и примеры (10–25 минут)

В HuggingFace `datasets` NER-датасеты обычно хранятся так:
- `tokens`: список токенов-слов (до subword-токенизации)
- `ner_tags`: список целочисленных меток такой же длины

Мы попробуем загрузить `conll2003` (английский NER).
Если он недоступен, используем небольшой встроенный набор.

In [ ]:

from datasets import Dataset, DatasetDict

def build_toy_ner_dataset():
    # Мини-датасет в стиле CoNLL: токены + BIO-метки
    # Типы: PER, ORG, LOC
    examples = [
        {
            "tokens": ["John", "works", "at", "OpenAI", "in", "San", "Francisco", "."],
            "ner_tags": ["B-PER", "O", "O", "B-ORG", "O", "B-LOC", "I-LOC", "O"]
        },
        {
            "tokens": ["Mary", "moved", "to", "Berlin", "last", "year", "."],
            "ner_tags": ["B-PER", "O", "O", "B-LOC", "O", "O", "O"]
        },
        {
            "tokens": ["Google", "hired", "Alice", "from", "London", "."],
            "ner_tags": ["B-ORG", "O", "B-PER", "O", "B-LOC", "O"]
        },
        {
            "tokens": ["Bob", "visited", "New", "York", "City", "."],
            "ner_tags": ["B-PER", "O", "B-LOC", "I-LOC", "I-LOC", "O"]
        },
    ]
    # Разобьём на train/validation/test
    train = Dataset.from_list(examples[:3])
    valid = Dataset.from_list(examples[3:4])
    test = Dataset.from_list(examples[3:4])
    return DatasetDict(train=train, validation=valid, test=test)

ds = None
try:
    from datasets import load_dataset
    ds = load_dataset("conll2003")
    print("Загружен conll2003:", ds)
except Exception as e:
    print("Не удалось загрузить conll2003, используем toy-датасет.")
    print("Причина:", repr(e))
    ds = build_toy_ner_dataset()

ds["train"][0]

## Словарь меток

Нам нужен список меток (`label_list`) и отображения:
- label → id
- id → label

Для `conll2003` метки уже закодированы числами и имеют `features`.
Для toy-датасета метки строковые — закодируем сами.

In [ ]:

from typing import List, Dict

def infer_label_list(dataset: DatasetDict) -> List[str]:
    # 1) Если датасет "как conll2003" — используем features
    if "ner_tags" in dataset["train"].features:
        feat = dataset["train"].features["ner_tags"]
        if hasattr(feat, "feature") and hasattr(feat.feature, "names"):
            return list(feat.feature.names)  # например: ["O","B-PER",...]
    # 2) Иначе соберём из строк
    labels = set()
    for split in ["train", "validation", "test"]:
        for ex in dataset[split]:
            for t in ex["ner_tags"]:
                labels.add(t)
    # Принято ставить "O" первым
    labels = sorted(labels)
    if "O" in labels:
        labels.remove("O")
        labels = ["O"] + labels
    return labels

label_list = infer_label_list(ds)
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

print("Количество меток:", len(label_list))
print(label_list[:20])

# 3. Теория: subword-токенизация и выравнивание меток (25–35 минут)

BERT использует subword-токенизацию (WordPiece/BPE): одно слово может разбиться на несколько под-токенов.
Но метки NER заданы на уровне "слов".

Нужно **выравнять** метки:
- специальным токенам ([CLS], [SEP], padding) ставим `-100` (чтобы loss их игнорировал),
- первому subword-токену слова ставим метку слова,
- остальным subword-токенам обычно ставят либо:
  - `-100` (игнорировать), либо
  - ту же метку, что и слову (часто для NER используют повторение I-меток).

Мы будем использовать распространённый вариант:
- метка только на первом subword-токене,
- остальные subword-токены получают `-100`.

# 4. Пример: токенизатор и функция выравнивания меток (35–45 минут)

Мы используем `AutoTokenizer` и `AutoModelForTokenClassification`.

Модель по умолчанию: `bert-base-cased` (английский) или `bert-base-multilingual-cased` (мультиязычная).
Если модель недоступна для скачивания, укажите локальный путь к модели в переменной `MODEL_NAME_OR_PATH`.

In [ ]:

from transformers import AutoTokenizer

MODEL_NAME_OR_PATH = "bert-base-cased"  # можно заменить на "bert-base-multilingual-cased" или локальный путь

tokenizer = None
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, use_fast=True, local_files_only=True)
    print("Токенизатор найден локально:", MODEL_NAME_OR_PATH)
except Exception:
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, use_fast=True)
        print("Токенизатор скачан:", MODEL_NAME_OR_PATH)
    except Exception as e:
        raise RuntimeError(
            "Не удалось получить токенизатор. Укажите локальный путь в MODEL_NAME_OR_PATH "
            "или обеспечьте доступ к интернету для скачивания модели."
        ) from e

tokenizer.is_fast

In [ ]:

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    # Для conll2003 ner_tags могут быть уже int; для toy — строки
    raw_tags = examples["ner_tags"]

    aligned_labels = []
    for i in range(len(raw_tags)):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                tag = raw_tags[i][word_id]
                if isinstance(tag, str):
                    label_ids.append(label2id[tag])
                else:
                    label_ids.append(int(tag))
            else:
                label_ids.append(-100)
            prev_word_id = word_id
        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

tokenized_ds = ds.map(tokenize_and_align_labels, batched=True)
tokenized_ds["train"][0].keys()

## Проверка выравнивания на одном примере

Мы распечатаем:
- исходные слова,
- subword-токены,
- итоговые label ids и их строковые имена (там, где label != -100).

In [ ]:

def show_alignment(example_idx=0):
    ex = ds["train"][example_idx]
    tok = tokenized_ds["train"][example_idx]

    input_ids = tok["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    labels = tok["labels"]

    print("Слова:", ex["tokens"])
    print()
    print("Subword-токены и метки:")
    for t, lid in zip(tokens, labels):
        if lid == -100:
            print(f"{t:>12} -> (ignore)")
        else:
            print(f"{t:>12} -> {id2label[int(lid)]}")

show_alignment(0)

# Самостоятельная работа 1 (5–7 минут)

Измените стратегию выравнивания так, чтобы:
- первому subword-токену ставилась исходная метка,
- остальным subword-токенам ставилась **та же метка**, но если это `B-XXX`, то для продолжений используйте `I-XXX`.

Подсказка: нужно преобразовывать `B-ORG → I-ORG`, `B-LOC → I-LOC` и т. п.

In [ ]:

# YOUR CODE HERE
# Реализуйте tokenize_and_align_labels_repeat_inside(examples)
# и примените его к датасету в переменной tokenized_ds_repeat

def to_inside(label_name: str) -> str:
    if label_name.startswith("B-"):
        return "I-" + label_name[2:]
    return label_name

def tokenize_and_align_labels_repeat_inside(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    raw_tags = examples["ner_tags"]

    aligned_labels = []
    for i in range(len(raw_tags)):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                tag = raw_tags[i][word_id]
                if isinstance(tag, str):
                    tag_name = tag
                else:
                    tag_name = label_list[int(tag)]

                if word_id != prev_word_id:
                    label_ids.append(label2id[tag_name])
                else:
                    tag_inside = to_inside(tag_name)
                    label_ids.append(label2id.get(tag_inside, label2id[tag_name]))
            prev_word_id = word_id
        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

tokenized_ds_repeat = ds.map(tokenize_and_align_labels_repeat_inside, batched=True)
print("Готово. Пример:")
show_alignment(0)

# 5. Fine-tuning BERT для NER (45–65 минут)

Теперь дообучим модель `AutoModelForTokenClassification`.

Мы используем:
- `DataCollatorForTokenClassification` для padding,
- `Trainer` для обучения,
- небольшое число эпох и ограничение на размер выборки (если вы хотите ускорить запуск).

In [ ]:

from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer

model = None
try:
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME_OR_PATH,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
        local_files_only=True
    )
    print("Модель найдена локально:", MODEL_NAME_OR_PATH)
except Exception:
    try:
        model = AutoModelForTokenClassification.from_pretrained(
            MODEL_NAME_OR_PATH,
            num_labels=len(label_list),
            id2label=id2label,
            label2id=label2id
        )
        print("Модель скачана:", MODEL_NAME_OR_PATH)
    except Exception as e:
        raise RuntimeError(
            "Не удалось получить модель. Укажите локальный путь в MODEL_NAME_OR_PATH "
            "или обеспечьте доступ к интернету."
        ) from e

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## Метрика seqeval

Для NER стандартная метрика — F1 по сущностям.
Мы используем `seqeval`. Важно:
- метки `-100` нужно исключать,
- предсказания модели нужно перевести из id в строки.

In [ ]:

import evaluate

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=-1)

    true_predictions = []
    true_labels = []
    for pred_seq, label_seq in zip(preds, labels):
        seq_preds = []
        seq_labels = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue
            seq_preds.append(id2label[int(pred_id)])
            seq_labels.append(id2label[int(label_id)])
        true_predictions.append(seq_preds)
        true_labels.append(seq_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    # Обычно интересны overall_precision/recall/f1/accuracy
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Быстро проверим формат метрики на маленьком примере (без обучения):
dummy_preds = np.zeros((2, 5, len(label_list)))
dummy_labels = np.full((2, 5), -100)
print("Метрика готова.")

## Настройки обучения

Для семинара достаточно 1–3 эпох.
Если датасет `conll2003` большой и вы хотите ускорить обучение, можно взять подвыборку.

In [ ]:

# Опционально: ускорение за счёт подвыборки (раскомментируйте при необходимости)
# tokenized_train = tokenized_ds["train"].shuffle(seed=SEED).select(range(2000))
# tokenized_valid = tokenized_ds["validation"].shuffle(seed=SEED).select(range(500))
# tokenized_test  = tokenized_ds["test"].shuffle(seed=SEED).select(range(500))

tokenized_train = tokenized_ds["train"]
tokenized_valid = tokenized_ds["validation"]
tokenized_test  = tokenized_ds["test"]

training_args = TrainingArguments(
    output_dir="./bert-ner-seminar",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

## Оценка на валидации и тесте

Мы посмотрим:
- итоговую F1,
- пример предсказаний на одном предложении.

In [ ]:

valid_metrics = trainer.evaluate()
print("Validation metrics:", valid_metrics)

test_metrics = trainer.evaluate(tokenized_test)
print("Test metrics:", test_metrics)

# Самостоятельная работа 2 (7–10 минут)

1) Увеличьте число эпох до 2–3 и сравните метрики.  
2) Измените `learning_rate` на 5e-5 и сравните.  
3) Сделайте вывод: что происходит с F1 и почему (переобучение/недообучение)?

Подсказка: при маленьком датасете toy-режим легко переобучается.

In [ ]:

# Эталонный вариант: проведите 2 эксперимента и сравните результаты.
# (В учебных целях мы оформляем это как "решение", но вы можете сначала сделать сами.)

def run_experiment(num_epochs=2, lr=5e-5):
    args = TrainingArguments(
        output_dir="./bert-ner-seminar-exp",
        learning_rate=lr,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        seed=SEED
    )
    exp_trainer = Trainer(
        model=AutoModelForTokenClassification.from_pretrained(
            MODEL_NAME_OR_PATH,
            num_labels=len(label_list),
            id2label=id2label,
            label2id=label2id
        ),
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    exp_trainer.train()
    return exp_trainer.evaluate()

# Закомментировано по умолчанию, чтобы не тратить время на запуск автоматически.
# metrics_e2_lr2e5 = run_experiment(num_epochs=2, lr=2e-5)
# metrics_e3_lr2e5 = run_experiment(num_epochs=3, lr=2e-5)
# metrics_e2_lr5e5 = run_experiment(num_epochs=2, lr=5e-5)
# print(metrics_e2_lr2e5, metrics_e3_lr2e5, metrics_e2_lr5e5)

# 6. Инференс и анализ ошибок (65–80 минут)

Сделаем предсказания на пользовательских примерах.
Мы будем:
- токенизировать предложение,
- получать logits,
- декодировать теги с учётом word_ids,
- выводить метки на уровне слов.

In [ ]:

from transformers import pipeline

ner_pipe = pipeline(
    "token-classification",
    model=trainer.model.to(device),
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if device == "cuda" else -1
)

text = "John works at OpenAI in San Francisco."
pred = ner_pipe(text)
pred[:5], len(pred)

## Интерпретация результата

`pipeline` возвращает найденные фрагменты (спаны) с типом сущности и уверенностью.
Для обучения и диагностики полезно дополнительно смотреть токен-уровневые предсказания на конкретных примерах из датасета.

In [ ]:

# Функция для токен-уровневого вывода (на уровне слов датасета)
def predict_tokens(tokens):
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        outputs = trainer.model(**inputs)
    preds = outputs.logits.argmax(dim=-1).squeeze(0).cpu().numpy().tolist()
    word_ids = inputs.word_ids(batch_index=0)

    out = []
    prev_word = None
    for tok_id, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id != prev_word:
            out.append((tokens[word_id], id2label[preds[tok_id]]))
        prev_word = word_id
    return out

print(predict_tokens(["Mary", "moved", "to", "Berlin", "."]))

# Самостоятельная работа 3 (5–7 минут)

1) Придумайте 3 предложения, содержащие людей/организации/локации.  
2) Прогоните через `ner_pipe` и через `predict_tokens`.  
3) Сравните, какие ошибки делает модель и почему.

Если вы используете `conll2003`, попробуйте предложения с редкими сущностями и необычными именами.

In [ ]:

# Эталонный пример анализа (замените на свои предложения)
examples = [
    "Alice joined Google in London.",
    "Bob lives in New York City.",
    "OpenAI released a model in San Francisco."
]

for s in examples:
    print("Текст:", s)
    print("Спаны:", ner_pipe(s))
    print()

# Итоги семинара

Вы проделали полный цикл fine-tuning для NER:
- подготовили данные и метки,
- учли subword-токенизацию и выравнивание меток,
- дообучили модель BERT для token classification,
- оценили качество по F1 и сделали инференс на своих примерах.

## Вопросы для самопроверки
1) Почему NER оценивают F1, а не accuracy?  
2) Зачем ставить `-100` на части subword-токенов?  
3) Что изменится, если размечать все subword-токены (B→I для продолжений)?  
4) Какие типы ошибок чаще: границы (segmentation) или тип (classification)?